# vLLM 前缀复用：开 vs 关 受控对照

## 为什么要补这一轮

2026-09-05 跑完 SGLang 之后发现一个**口径不对等**的问题：

| | 已有数字 | 实际是什么的差 |
|---|---|---|
| vLLM（旧） | TTFT −35%、吞吐 +2.3% | **同一服务内「有无共享前缀」** |
| SGLang（新） | 吞吐 1.54×、TTFT −73.4% | **特性「开 vs 关」** |

**这两个口径不同，并列比大小是错的。** 本 notebook 给 vLLM 补上缺的那一半：
同样跑 `--enable-prefix-caching` **关**的一组，使两个框架都能给出「特性开 vs 关」的收益。

对照矩阵（并发 8、32 请求，与 SGLang 那轮完全一致）：

|  | 无共享前缀 | 共享前缀 2380 字符 |
|---|---|---|
| `--enable-prefix-caching` **开** | | |
| `--enable-prefix-caching` **关** | | |

**「特性真实收益」= 共享前缀那一列的 开 ÷ 关。**

## 已知会踩的坑（来自 SGLang 那轮，先写下来）

- Colab 预装的 `transformers` 与推理框架常有版本冲突；本 notebook 装完会打印全部版本。
- `torchaudio` 与 `torch` 版本错配会让 transformers 的 `audio_utils` 炸掉整条导入链
  —— **不装 torchaudio**。
- vLLM 需要 `torchvision`（其 `model_executor` 会 import），**这个要装**，别照搬 SGLang 那份。


## 0. 环境

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout
print(out)
name = out.strip().splitlines()[-1].split(",")[0].strip()
print("显卡:", name)
print("Tensor Core:", "无（GTX 16 系）" if "GTX 16" in name else "有")
print()
print("要与 2026-09-05 SGLang 那轮并列，这里必须也是 Tesla T4。不是 T4 就别往下跑。")


## 1. 装 vLLM

装 `vllm` + `aiohttp` + `torchvision`（vLLM 的 model_executor 要 torchvision）。
**不装 `torchaudio`** —— SGLang 那轮实测它与 torch 版本错配会炸掉 transformers 的导入链。

判断条件是「三个包**全部**在场才跳过」。


In [ ]:
import importlib.metadata as md_, subprocess, sys

CHECK = ["vllm", "aiohttp", "torchvision"]

def ver(p):
    try:
        return md_.version(p)
    except Exception:
        return None

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=isinstance(cmd, str),
                          capture_output=True, text=True, **kw)

missing = [p for p in CHECK if ver(p) is None]
print("缺失:", missing or "无")

if missing:
    print("装 vllm + aiohttp + torchvision（一条命令，约 5-10 分钟）...")
    r = sh([sys.executable, "-m", "pip", "install", "-q", "vllm", "aiohttp", "torchvision"])
    print("退出码:", r.returncode)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
else:
    print("三个包都在，跳过安装。")

print()
print("清掉会炸导入链的 torchaudio（SGLang 那轮实测）...")
u = sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"])
print("  卸 torchaudio 退出码", u.returncode)

print()
print("=== 版本 ===")
for p in CHECK + ["transformers", "torch"]:
    print("  %-16s %s" % (p, ver(p) or "（未安装）"))
print("  %-16s %s" % ("torchaudio", ver("torchaudio") or "已卸载 ✓"))


## 2. 写出压测脚本（与 SGLang 那轮同一份，一个字节不改）

In [ ]:
import io
src = '# -*- coding: utf-8 -*-\n"""vLLM 服务端压测：并发扫描下的吞吐 / TTFT / TPOT，以及前缀复用的效果。\n\n指标定义（与 JD 里那套一致）：\n  TTFT  Time To First Token   —— 首 token 延迟，决定交互体感\n  TPOT  Time Per Output Token —— 首 token 之后的平均出词间隔\n  吞吐   总输出 token 数 / 墙钟时间\n\n用法（先 bash serve.sh 起服务）：\n  python bench_serving.py                 # 并发扫描\n  python bench_serving.py --prefix-test   # 前缀复用对照\n"""\nimport argparse, asyncio, json, statistics as st, time\nimport aiohttp\n\nURL = "http://127.0.0.1:8000/v1/chat/completions"\nMODEL = "Qwen/Qwen2.5-0.5B-Instruct"\n\n# 一段较长的共享 system prompt：开 --enable-prefix-caching 后其 prefill 只算一次\nSHARED_PREFIX = (\n    "You are a meticulous technical assistant. Answer concisely and precisely. "\n    "Always reason step by step before answering. " * 20\n)\n\n\nasync def one_request(sess, prompt, max_tokens, use_prefix):\n    msgs = ([{"role": "system", "content": SHARED_PREFIX}] if use_prefix else []) + \\\n           [{"role": "user", "content": prompt}]\n    body = {"model": MODEL, "messages": msgs, "max_tokens": max_tokens,\n            "temperature": 0.0, "stream": True}\n    t0 = time.perf_counter()\n    ttft, n_tok, last = None, 0, t0\n    async with sess.post(URL, json=body) as resp:\n        async for raw in resp.content:\n            line = raw.decode("utf-8").strip()\n            if not line.startswith("data: ") or line == "data: [DONE]":\n                continue\n            delta = json.loads(line[6:])["choices"][0].get("delta", {})\n            if delta.get("content"):\n                now = time.perf_counter()\n                if ttft is None:\n                    ttft = now - t0\n                n_tok += 1\n                last = now\n    return dict(ttft=ttft or 0.0, total=last - t0, n_tok=n_tok)\n\n\nasync def run_batch(n_conc, n_req, max_tokens, use_prefix):\n    prompts = [f"Explain concept #{i} in distributed systems." for i in range(n_req)]\n    sem = asyncio.Semaphore(n_conc)\n\n    async def guarded(sess, p):\n        async with sem:\n            return await one_request(sess, p, max_tokens, use_prefix)\n\n    timeout = aiohttp.ClientTimeout(total=600)\n    async with aiohttp.ClientSession(timeout=timeout) as sess:\n        await one_request(sess, "warmup", 4, use_prefix)          # 预热\n        t0 = time.perf_counter()\n        rs = await asyncio.gather(*(guarded(sess, p) for p in prompts))\n        wall = time.perf_counter() - t0\n\n    tot_tok = sum(r["n_tok"] for r in rs)\n    tpots = [(r["total"] - r["ttft"]) / max(r["n_tok"] - 1, 1) for r in rs if r["n_tok"] > 1]\n    return dict(conc=n_conc, wall=wall, tput=tot_tok / wall, rps=len(rs) / wall,\n                ttft_p50=st.median(r["ttft"] for r in rs),\n                ttft_p99=sorted(r["ttft"] for r in rs)[int(len(rs) * 0.99) - 1],\n                tpot_p50=st.median(tpots) if tpots else 0.0, tot_tok=tot_tok)\n\n\nasync def sweep(args):\n    print(f"{\'并发\':>5}{\'请求\':>6}{\'墙钟s\':>9}{\'吞吐 tok/s\':>13}{\'RPS\':>8}"\n          f"{\'TTFT p50\':>11}{\'TTFT p99\':>11}{\'TPOT p50\':>11}")\n    print("-" * 74)\n    out = []\n    for c in [1, 2, 4, 8, 16, 32]:\n        r = await run_batch(c, max(c * 4, 16), args.max_tokens, use_prefix=False)\n        print(f"{r[\'conc\']:>5}{max(c*4,16):>6}{r[\'wall\']:>9.2f}{r[\'tput\']:>13.1f}"\n              f"{r[\'rps\']:>8.2f}{r[\'ttft_p50\']*1e3:>10.1f}ms{r[\'ttft_p99\']*1e3:>10.1f}ms"\n              f"{r[\'tpot_p50\']*1e3:>10.2f}ms")\n        out.append(r)\n    json.dump(out, open("sweep_results.json", "w"), indent=1)\n    base = out[0]["tput"]\n    print(f"\\ncontinuous batching 收益：并发 1 → 32，吞吐 "\n          f"{base:.1f} → {out[-1][\'tput\']:.1f} tok/s（{out[-1][\'tput\']/base:.1f}×），"\n          f"TTFT p50 {out[0][\'ttft_p50\']*1e3:.0f} → {out[-1][\'ttft_p50\']*1e3:.0f} ms")\n    print("吞吐与延迟的取舍就在这张表里：并发拉高吞吐涨，但 TTFT 同步恶化。")\n\n\nasync def prefix_test(args):\n    print("前缀复用对照（服务端需带 --enable-prefix-caching 启动）")\n    print(f"{\'场景\':<26}{\'吞吐 tok/s\':>13}{\'TTFT p50\':>12}")\n    print("-" * 51)\n    for label, up in [("无共享前缀", False), (f"共享前缀 ({len(SHARED_PREFIX)} 字符)", True)]:\n        r = await run_batch(8, 32, args.max_tokens, use_prefix=up)\n        print(f"{label:<26}{r[\'tput\']:>13.1f}{r[\'ttft_p50\']*1e3:>11.1f}ms")\n    print("\\n共享前缀命中 KV cache 后，重复的 prefill 不再重算，TTFT 应显著下降。")\n    print("对比未开 --enable-prefix-caching 重启服务再跑一次，差值即为该特性的真实收益。")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--max-tokens", type=int, default=128)\n    ap.add_argument("--prefix-test", action="store_true")\n    a = ap.parse_args()\n    asyncio.run(prefix_test(a) if a.prefix_test else sweep(a))\n'
io.open("bench_serving.py", "w", encoding="utf-8").write(src)
print("写出 bench_serving.py", len(src), "字符")
print("SHA 前 16 位（与 SGLang 那轮比对用）:")
import hashlib
print(" ", hashlib.sha256(src.encode("utf-8")).hexdigest()[:16])


## 3. 启动器

与 SGLang 那轮同样的做法：起不来就把失败原因原样打出来，不静默重试。
`--gpu-memory-utilization 0.80` 对齐 SGLang 的 `--mem-fraction-static 0.80`。


In [ ]:
import subprocess, sys, time, requests, re

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def serve_vllm(extra, tag, wait=420):
    subprocess.run(["pkill", "-f", "vllm"], check=False)
    time.sleep(10)
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", MODEL,
           "--host", "127.0.0.1", "--port", "8000",
           "--max-model-len", "2048",
           "--gpu-memory-utilization", "0.80",
           "--disable-log-requests"] + extra
    print("启动参数:", " ".join(cmd[2:]))
    print()
    log = open("/content/vllm_%s.log" % tag, "w")
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

    for i in range(wait // 2):
        if p.poll() is not None:
            print("[%s] 进程已退出（码 %s），日志尾部：" % (tag, p.returncode))
            print(open("/content/vllm_%s.log" % tag).read()[-4000:])
            return None
        try:
            if requests.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
                print("[%s] 就绪，用时 %ds" % (tag, i * 2))
                txt = open("/content/vllm_%s.log" % tag).read()
                for pat in ["GPU KV cache size", "Maximum concurrency",
                            "model weights take", "Prefix caching", "prefix_caching"]:
                    for ln in txt.splitlines():
                        if pat.lower() in ln.lower():
                            print("   ", ln.strip()[:170])
                            break
                return p
        except requests.RequestException:
            pass          # 只吞连接失败；其它异常照抛，别再无声吞掉真 bug
        time.sleep(2)

    print("[%s] %ds 内没起来，日志尾部：" % (tag, wait))
    print(open("/content/vllm_%s.log" % tag).read()[-4000:])
    return None


## 4. 前缀复用 **开**

`--enable-prefix-caching`。这一组是已有数据的重跑，用来确认本次环境与上一轮可比。


In [ ]:
proc = serve_vllm(["--enable-prefix-caching"], "prefix_on")

In [ ]:
!python -u bench_serving.py --prefix-test

## 5. 前缀复用 **关**（本轮要补的那一半）

`--no-enable-prefix-caching`。若该参数在当前 vLLM 版本里不叫这个名字，下一格会直接报错——
**不要猜参数名**，把报错发我。


In [ ]:
import subprocess, sys, re
r = subprocess.run([sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--help"],
                   capture_output=True, text=True)
h = r.stdout + r.stderr
print("help returncode:", r.returncode, "| 长度:", len(h))
print("含 --no-enable-prefix-caching:", "--no-enable-prefix-caching" in h)
print("含 --enable-prefix-caching   :", "--enable-prefix-caching" in h)
print("所有 prefix 相关参数:", sorted(set(re.findall(r"--[a-z0-9-]*prefix[a-z0-9-]*", h))))


In [ ]:
proc = serve_vllm(["--no-enable-prefix-caching"], "prefix_off")

In [ ]:
!python -u bench_serving.py --prefix-test

## 6. 怎么算，怎么写

把四个数填进去：

|  | 无共享前缀 吞吐 / TTFT | 共享前缀 吞吐 / TTFT |
|---|---|---|
| 开 | | |
| 关 | | |

**vLLM 前缀复用的真实收益 = 共享前缀那一列：开 ÷ 关。**
算完与 SGLang 的 **1.54× / TTFT −73.4%** 并列——**这时候才是同口径，才能比。**

### 顺带要看的第二件事

SGLang 那轮发现：**没有**共享前缀时 RadixAttention 是净开销（TTFT 恶化 45%）。
vLLM 这边「无共享前缀」那一列的 开 vs 关，能回答同一个问题——
**vLLM 的 prefix caching 在无复用场景下是否也要付代价。**
不管答案是哪边，照实写。

### 边界

- 与 SGLang 那轮同硬件（T4）、同模型、同一份 `bench_serving.py`（第 2 节打了 SHA 供比对）。
- vLLM 走默认后端，SGLang 走 Triton + PyTorch（非其默认最优路径）——
  **纵向比两个框架绝对值时必须带上这条**，横向比各自的「开 vs 关」不受影响。
- 单次测量、无重复跑、无置信区间。SGLang 那轮出现过两处未复现的离群值，
  本轮若也出现，同样如实记、不解释、不引用。
